# Week 6 Coding: solutions

Three tasks, each connected to the problem set: **C1** ↔ P6/P8/P9, **C2** ↔ P2/P3, **C3** ↔ P4 (optional).

**How to access and work with this solution notebook.** Open this notebook through the corresponding Moodle solution activity. If prompted, click **“Start My Server”** and wait for JupyterLab to load. Run cells top to bottom with Shift+Enter. A passing check prints a confirmation; a failing one raises an `AssertionError`. If the notebook gets into a confused state, use **Kernel → Restart Kernel**, then run the cells again from the top. This notebook contains the completed versions of C1–C3 for comparison with your work.

## C1: Folding, and a check by sampling *(↔ P6, P8, P9)*

Every CT sinusoid has the same samples as exactly one sinusoid with frequency in $[0, f_s/2]$ (P11). The function `fold(f, fs)` returns that frequency.

a) Complete `fold`: first remove the multiples of $f_s$ (P11 a), then reflect about $f_s/2$ (P11 b).

b) Before running the check cell, write down `fold(15, 12)`, `fold(9, 12)` and `fold(10, 12)` (P6, P8 d). The check cell also verifies by sampling that a sinusoid at `f` and one at `fold(f, fs)` give the same samples.

In [ ]:
import numpy as np

def fold(f, fs):
    """Frequency in [0, fs/2] whose samples coincide with those of a sinusoid at f."""
    r = f % fs                  # remove the multiples of fs (P11 a)
    return min(r, fs - r)       # reflect about fs/2 (P11 b)

In [ ]:
assert fold(15, 12) == 3                                                    # P6: the hum
assert fold(9, 12) == 3 and fold(10, 12) == 2                               # P8 d
assert [f for f in range(30) if fold(f, 10) == 3] == [3, 7, 13, 17, 23, 27]  # P9 a
n, fs = np.arange(40), 12.0
for f in (9.0, 10.0, 15.0, 27.0):
    assert np.allclose(np.cos(2*np.pi*f*n/fs), np.cos(2*np.pi*fold(f, fs)*n/fs))
print("C1 passed: fold matches P6, P8 d and P9 a, and the folded sinusoid has the same samples.")

In [ ]:
import matplotlib.pyplot as plt

fs, f = 12.0, 10.0                              # kHz: the example of P8 d
t = np.linspace(0, 1, 2000)                     # time in ms
n = np.arange(0, 13)                            # samples at t = n / fs
plt.figure(figsize=(9, 2.6))
plt.plot(t, np.cos(2*np.pi*f*t), lw=0.8, label=f"{f:g} kHz")
plt.plot(t, np.cos(2*np.pi*fold(f, fs)*t), lw=2, label=f"{fold(f, fs):g} kHz (folded)")
plt.plot(n/fs, np.cos(2*np.pi*f*n/fs), "ko", label="samples")
plt.xlabel("$t$ [ms]"); plt.legend(fontsize=8, loc="upper right"); plt.tight_layout()

## C2: A signal built from its components *(↔ P2, P3)*

A sum of sinusoids is described by a list of components `(A, Omega, phi)`, one per sinusoid $A\cos(\Omega n + \varphi)$; a constant is the component with $\Omega = 0$.

a) Complete `from_components`, which returns the samples $x[0], \dots, x[N-1]$ of the sum.

b) Before running the check cell, write down the first three samples of P2's signal. The check cell verifies them and the period, and compares the two signals of P3, which have the same amplitudes and frequencies but different phases.

In [ ]:
def from_components(components, N):
    """x[n] = sum of A cos(Omega n + phi) over the components, for n = 0, ..., N-1."""
    n = np.arange(N)
    return sum(A * np.cos(W * n + phi) for A, W, phi in components)

x  = from_components([(1, 0, 0), (4, 2*np.pi/3, 0)], 9)          # P2
x1 = from_components([(1, np.pi/4, 0), (1, np.pi/2, 0)], 8)      # P3: x1
x2 = from_components([(1, np.pi/4, 0), (1, np.pi/2, np.pi)], 8)  # P3: x2 (minus sign = phase pi)
print(np.round(x, 6)); print(np.round(x1, 3)); print(np.round(x2, 3))

In [ ]:
assert np.allclose(x[:3], [5, -1, -1])                 # P2: the samples
assert np.allclose(x[3:], x[:-3])                      # P2: period 3
assert np.isclose(x1[0], 2) and np.isclose(x2[0], 0)   # P3: the sequences differ ...
assert not np.allclose(x1, x2)                         # ... although the amplitudes are equal
print("C2 passed: P2's samples and period; equal amplitudes do not imply equal signals (P3).")

## C3: The pitch of a sampled frequency sweep *(↔ P4)* (optional)

A sinusoid whose frequency rises from $0$ to $12$ kHz over $12$ s is sampled with $f_s = 8$ kHz. Its pitch is measured from the samples and compared with the prediction of P4, `fold(f, fs)`.

a) Complete `measured_pitch`: count the sign changes of the samples in a window of `width` seconds around `t0`; a sinusoid at $f$ changes sign $2f$ times per second.

b) Before running the check cell, write down the predicted pitch at $t = 2, 4, 6, 10$ s. Then run the check cell and the plot cell; the last cell plays the sampled sweep.

In [ ]:
fs, T, f_end = 8000, 12.0, 12000.0
t = np.arange(0, T, 1/fs)
x = np.cos(np.pi * (f_end/T) * t**2)          # instantaneous frequency (f_end/T) t

def measured_pitch(x, t0, fs, width=0.1):
    """Pitch in Hz from the sign changes of the samples in [t0 - width/2, t0 + width/2]."""
    seg = x[int((t0 - width/2)*fs):int((t0 + width/2)*fs)]
    crossings = np.sum(np.signbit(seg[1:]) != np.signbit(seg[:-1]))
    return crossings / width / 2

for t0 in (2, 4, 6, 10):
    print(f"t = {t0:2d} s:  measured {measured_pitch(x, t0, fs):6.0f} Hz,  predicted {fold(f_end/T*t0, fs):6.0f} Hz")

In [ ]:
for t0 in (2, 4, 6, 10):
    predicted = fold(f_end/T*t0, fs)
    assert abs(measured_pitch(x, t0, fs) - predicted) < 0.03 * fs    # within 3 % of fs
assert fold(f_end/T*8, fs) == 0                                      # t = 8 s: the pitch returns to 0
print("C3 passed: the measured pitch follows the folded frequency, rising, falling and rising again (P4).")

In [ ]:
times = np.arange(0.5, 11.6, 0.25)
plt.figure(figsize=(9, 2.6))
plt.plot(times, [f_end/T*t0 for t0 in times], ":", color="gray", label="frequency of the CT sinusoid")
plt.plot(times, [fold(f_end/T*t0, fs) for t0 in times], lw=2, label="predicted pitch: fold(f, fs)")
plt.plot(times, [measured_pitch(x, t0, fs) for t0 in times], "k.", label="measured from the samples")
plt.axhline(fs/2, color="gray", lw=0.6)
plt.xlabel("$t$ [s]"); plt.ylabel("Hz"); plt.legend(fontsize=8); plt.tight_layout()

In [ ]:
from IPython.display import Audio
Audio(0.3 * x, rate=fs)      # the sampled sweep: listen for the pitch rising, falling and rising again